In [ ]:
# RNN을 이용한 텍스트 생성
# 문맥을 반영해 다음 단어를 예측하여 텍스트 생성 (다항 분류)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences, to_categorical
import numpy as np
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.models import Sequential

# text = """경마장에 있는 말이 뛰고 있다
# 그의 말이 법이다
# 가는 말이 고와야 오는 말이 곱다"""

text = """
미국 행정부가 민간기업 투자를 중심으로 이란 재건에 나설 것으로 알려지면서 국내 산업계도 촉각을 기울이고 있다. 특히 세계적인 건설 및 에너지 플랜트 산업을 보유한 한국에 대한 재건 사업 참여 요구가 커질 것이란 분석이 나온다.
이란 재건뿐 아니라 전쟁으로 파괴된 중동 에너지 시설 복구 사업에 한국 기업이 대거 참여할 것으로 전망된다. 피해를 입은 카타르 라스라판 액화천연가스(LNG) 시설 등 중동 에너지 생산 인프라 상당수를 한국 건설사들이 시공해 왔기 때문이다. 전쟁으로 파괴된 중동 일대 에너지 인프라 복구 비용만 약 88조 원 규모로 추산된다.
기업, 중동 에너지 인프라 복구 나설 듯
종전 합의 소식이 알려진 15, 16일 이틀간 국내 주식시장에서는 대우건설 주가가 20.4%, DL이앤씨 주가가 17.3% 오르는 등 건설주가 상승세를 보였다. 국내 건설사들이 중동 에너지 인프라 복구와 이란 재건 사업에 참여할 수 있다는 시장의 기대가 커진 것이다.
실제로 이란을 비롯한 중동 지역은 원유·가스 관련 설비를 비롯해 발전소, 송배전망, 항만, 도로, 철도, 통신망 등 국가 핵심 기반시설 복구를 위해 대규모 투자에 나설 전망이다. 전쟁으로 이란의 정유 시설뿐만 아니라 아랍에미리트(UAE) 루와이스 정유시설, 카타르 라스라판 LNG 처리시설 등 중동 각국의 에너지 생산 인프라가 타격을 입었다. 글로벌 에너지 공급망의 안정화를 위해서라도 빠른 복구가 절실한 상황이다. 노르웨이 에너지 컨설팅 업체 리스타드에너지는 이번 중동 전쟁으로 인한 중동 에너지 인프라 복구 비용이 약 580억 달러(약 87조7800억 원)에 이를 것으로 전망했다.
특히 피해를 입은 카타르나 UAE의 가스·정유시설 등은 국내 건설사가 2000년대 후반 대거 수주해 설계나 시설 구조를 이미 파악하고 있다는 게 강점으로 꼽힌다. 카타르 라스라판 LNG 시설은 삼성E&A가, UAE 합샨의 가스 처리 시설은 현대건설이 각각 시공했다. 현재도 이란 현지에 사무소를 유지하고 있는 DL이앤씨는 버락 오바마 정부가 대이란 경제 제재를 해제한 2017년 당시 이란 이스파한 정유공장 개선 공사를 수주한 바 있다.
재건 사업에 투입될 건설기계 수요도 크게 늘어나 HD건설기계와 두산밥캣 등 중동 지역에 공을 들여 온 국내 건설기계 업체의 대규모 수출도 성사될 것이라는 관측이 나온다.
전쟁으로 파괴된 전력, 통신망 재구축과 더불어 노후 망 교체 작업도 이뤄질 전망인 만큼 관련 역량을 가진 LS전선, 대한전선 등 국내 전선업계의 참여도 거론된다. 정유·석유화학 시설 재구축에 HD현대일렉트릭, 효성중공업, LS일렉트릭 등 국내 업체의 변압기와 배전반 등이 쓰일 가능성도 높다.
국내 정유사들은 종전에 따른 원유 공급망 정상화를 기대하고 있다. 제재 완화 정도에 따라 저렴하고 질 좋은 이란산 원유를 활용할 수 있게 된다면 공급망 다변화 측면에서 선택지를 넓힐 수 있다.
 정세 급변 리스크 여전… 美 청구서도 변수
다만 이란이 아직 미국의 경제 제재를 받고 있는 데다 제재가 풀리더라도 중동 정세가 언제든지 급변할 수 있다는 점은 여전한 리스크로 꼽힌다. 한국 건설사들은 과거 중동 재건사업에 참여했다 현지 상황이 급변하며 미수금 문제가 발생하는 등 어려움을 겪었었다.
미국의 재건기금은 아직 구체적인 운용 방식이 알려지지 않은 상태다. 한국 민관이 자금을 투자해 복구 사업에 참여해야 하는 ‘비싼’ 청구서로 돌아온다면 부담만 커질 것이란 지적도 나온다.
손태홍 한국건설산업연구원 건설기술·관리 연구실장은 “미국의 재건기금 조성 여부와 상관없이 재건 시장이 열릴 것이란 점은 확실하다”며 “국제사회의 재건기금 조성이 원활하게 이뤄질 수 있을지, 또 현지 기관의 사업관리 능력이 충분한지 등을 고려해 접근할 필요가 있다”고 조언했다.
"""

# tok = Tokenizer(char_level=True)    # 글자 단위
# tok = Tokenizer(char_level=False)   # 단어 단위. 기본 값
tok = Tokenizer()
tok.fit_on_texts([text])
encoded = tok.texts_to_sequences([text])[0]
print(encoded)
print(tok.word_index)

vocab_size = len(tok.word_index) + 1  # 실제 단어 집합 + 1을 함

# 훈련 데이터 작성
sequences = list()
for line in text.split('\n'):  # 문장 토큰화
    enco = tok.texts_to_sequences([line])[0]
    # print(enco)
    # 바로 다음 단어를 label로 사용하기 위해 리스트에 담기
    for i in range(1, len(enco)):
        sequ = enco[:i + 1]
        # print(sequ)
        sequences.append(sequ)

print('학습에 참여할 샘플 수 : ', len(sequences))   # 11
print(sequences)
print(max(len(i) for i in sequences))

# 전체 각각의 벡터 길이를 통일
max_len = max(len(i) for i in sequences)
psequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
print(psequences)

print()
# 각 벡터의 마지막 요소를 label로 사용하기 위해 분리
x = psequences[:, :-1]   # feature
y = psequences[:, -1]    # label
print(x)
print(y)

# label은 one-hot encoding 필요
y = to_categorical(y, num_classes=vocab_size)
print(y[:2])

In [ ]:
# model
model = Sequential()
model.add(Embedding(vocab_size, 32, input_length=max_len - 1))
model.add(LSTM(32, activation='tanh'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

model.fit(x, y, epochs=200, verbose=2)
print(model.evaluate(x, y))

In [ ]:
# 문자열 생성 함수
def sequence_gen_text(model, t, current_word, n):
    init_word = current_word
    sentence = ""
    for _ in range(n):
        encoded = t.texts_to_sequences([current_word])[0]
        encoded = pad_sequences([encoded], maxlen=max_len - 1, padding='pre')
        result = np.argmax(model.predict(encoded, verbose=0), axis=-1)

        # 예측 단어 찾기
        for word, index in t.word_index.items():
            # print(word, index)
            if index == result:
                break

        current_word = current_word + ' ' + word
        sentence = sentence + ' ' + word

    sentence = init_word + sentence
    return sentence

# print(sequence_gen_text(model, tok, '경마장', 5))
# print(sequence_gen_text(model, tok, '그의', 5))
# print(sequence_gen_text(model, tok, '고와야', 5))
print(sequence_gen_text(model, tok, '미국 행정부', 20))
print(sequence_gen_text(model, tok, '제재', 20))
print(sequence_gen_text(model, tok, '세계적인', 20))